### SmallTalk 3: Final Evaluation
Runs the fine-tuned models, scores every run the same way, and answers the main question: how much of the gap to the bigger models did LoRA SFT and DPO close?

### Setup
On Colab, choose **Runtime > Change runtime type > T4 GPU** first. Upload `smalltalk.py`, `config.py` and `requirements.txt` to `MyDrive/smalltalk/` (or keep them next to this notebook in VS Code).

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/smalltalk'  # results are saved on Drive so a disconnect does not lose them
else:
    WORK = os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)   # so the notebook can import smalltalk.py and config.py
print('Working folder:', WORK)
for f in ['smalltalk.py', 'config.py', 'requirements.txt']:
    assert os.path.exists(f), f'Put {f} in the working folder first'

In [ ]:
%pip install -q -r requirements.txt

### Config
All settings (models, hyperparameters, prompts) are in `config.py`. Edit that file, save it, and run this cell again. To change a setting for this notebook only, set it in a cell after this one.

`RUN_ON` sets where the notebook runs: `'local'` (6GB laptop), `'colab'` (T4), `'server'` (remote GPU server) or `'auto'` to detect it. It only changes the batch size and whether the 8B/9B models and training are run, so the results stay comparable.

`DOWNLOADS` is the one folder that holds everything downloaded: the dataset in `QMSum/` and the base models in `models/`. What is already there is reused, and only what is missing is downloaded.

In [ ]:
import json, glob, importlib
import pandas as pd
import config, smalltalk as st
importlib.reload(config); importlib.reload(st)   # picks up edits to config.py without a kernel restart
from config import *                              # every setting lives in config.py

st.set_seed(SEED)
os.makedirs('runs', exist_ok=True); os.makedirs('results', exist_ok=True)

PROFILE = st.run_profile(RUN_ON)
BATCH = PROFILE['batch_size']
# reuses the dataset and models that are already in the folder, downloads what is missing
st.setup_downloads(DOWNLOADS or PROFILE['downloads'])
# only needed if a model asks you to accept its terms on Hugging Face (keep it after setup_downloads)
# from huggingface_hub import login; login()

train_meetings, val_meetings, test_meetings = (st.load_split(s) for s in ['train', 'val', 'test'])
meetings_by_id = {m['id']: m for m in train_meetings + val_meetings + test_meetings}
test_examples = st.test_subset(st.build_examples(test_meetings), N_TEST)
print('Test questions used:', len(test_examples), '| K =', K, '| 4-bit:', LOAD_4BIT)

### 1. Run the fine-tuned models
Same test questions, same retriever, same agent as the baselines.

In [ ]:
retriever = st.Retriever()
for key, model_id in SMALL_MODELS.items():
    for stage in ['sft', 'dpo']:
        adapter = f'adapters/{key}_{stage}'
        if not os.path.exists(adapter):
            print('Skipping, no adapter yet:', adapter); continue
        llm = st.LLM(model_id, adapter=adapter, load_4bit=LOAD_4BIT)
        for mode in ['rag', 'agent']:
            st.run_experiment(llm, mode, test_examples, meetings_by_id, retriever, f'runs/{key}_{stage}_{mode}.jsonl', k=K, batch_size=BATCH)
        llm.unload()

### 2. Score every run
- **ROUGE-1/2/L**: word overlap with the human summary
- **BERTScore**: meaning overlap with the human summary
- **Faithfulness** (HHEM-2.1-Open): is the summary supported by the text the model read? A score under 0.5 counts as a hallucination
- **Tool use rate / tool calls / format errors**: how the agent behaved

In [ ]:
faith = st.Faithfulness()
results = []
for path in sorted(glob.glob('runs/*.jsonl')):
    key, stage, mode = st.parse_run_name(os.path.basename(path)[:-6])
    results.append({'model': key, 'stage': stage, 'mode': mode, **st.evaluate_run(path, faith)})
res = pd.DataFrame(results).drop(columns='run')
res.to_csv('results/all_results.csv', index=False)
res.round(3)

### 3. Did the 4B models close the gap?
`gap closed = (tuned 4B - base 4B) / (big model - base 4B)`. 0 means no progress, 1 means the 4B matched the big model, above 1 means it beat it. Agentic RAG is the main setting. This only makes sense when the big model scores higher than the base 4B; if the base 4B is already better on a metric, read the raw scores instead.

In [ ]:
def score(model, stage, mode, metric):
    r = res[(res['model'] == model) & (res['stage'] == stage) & (res['mode'] == mode)]
    return float(r[metric].iloc[0]) if len(r) else float('nan')

MODE = GAP_MODE   # set in config.py
rows = []
for small, big in FAMILY.items():
    for rival in [big, CROSS_FAMILY]:
        for metric in ['faithfulness', 'rougeL', 'bertscore']:
            base, sft, dpo, top = (score(small, 'base', MODE, metric), score(small, 'sft', MODE, metric),
                                   score(small, 'dpo', MODE, metric), score(rival, 'base', MODE, metric))
            rows.append({'small model': small, 'compared with': rival, 'metric': metric, 'base 4B': base,
                         '+SFT': sft, '+SFT+DPO': dpo, 'big model': top,
                         'gap closed (SFT)': st.gap_closed(base, sft, top), 'gap closed (DPO)': st.gap_closed(base, dpo, top)})
gap = pd.DataFrame(rows)
gap.to_csv('results/gap_closed.csv', index=False)
gap.round(3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# one panel per metric, bars: base 4B, tuned 4B (SFT+DPO), same-family big model
metrics = ['faithfulness', 'rougeL', 'bertscore']
colors = {'base 4B': '#2a78d6', '4B + SFT + DPO': '#eb6834', 'big model': '#1baf7a'}
fig, axes = plt.subplots(1, len(metrics), figsize=(13, 4))
for ax, metric in zip(axes, metrics):
    groups = list(FAMILY.items())
    x = np.arange(len(groups))
    vals = {'base 4B': [score(s, 'base', MODE, metric) for s, b in groups],
            '4B + SFT + DPO': [score(s, 'dpo', MODE, metric) for s, b in groups],
            'big model': [score(b, 'base', MODE, metric) for s, b in groups]}
    for i, (label, v) in enumerate(vals.items()):
        bars = ax.bar(x + (i - 1) * 0.26, v, width=0.24, color=colors[label], label=label)
        ax.bar_label(bars, fmt='%.2f', fontsize=8, padding=2)
    ax.set_xticks(x, [f'{s}\nvs {b}' for s, b in groups], fontsize=9)
    ax.set_title(metric); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, fontsize=10)
fig.suptitle(f'Agentic RAG results ({MODE} mode)')
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.savefig('results/main_chart.png', dpi=200); plt.show()

### 4. What did each step add?
Ablation for each small model: zero-shot, standard RAG, agentic RAG, then fine-tuning.

In [ ]:
order = [('base', 'zeroshot'), ('base', 'rag'), ('base', 'agent'), ('sft', 'rag'), ('sft', 'agent'), ('dpo', 'rag'), ('dpo', 'agent')]
abl = []
for small in SMALL_MODELS:
    for stage, mode in order:
        r = res[(res['model'] == small) & (res['stage'] == stage) & (res['mode'] == mode)]
        if len(r):
            abl.append({'model': small, 'setup': f'{stage} + {mode}',
                        **r[['rougeL', 'bertscore', 'faithfulness', 'hallucination_rate', 'avg_words']].iloc[0].to_dict()})
pd.DataFrame(abl).round(3)

### 5. General vs specific questions

In [ ]:
def by_qtype(path):
    rows = st.read_jsonl(path)
    out = {}
    for qtype in ['general', 'specific']:
        sub = [r for r in rows if r['qtype'] == qtype]
        if sub:
            out[f'rougeL_{qtype}'] = st.rouge([r['pred'] for r in sub], [r['answer'] for r in sub])['rougeL']
    return out

pd.DataFrame([{'run': os.path.basename(p)[:-6], **by_qtype(p)} for p in sorted(glob.glob('runs/*_agent.jsonl'))]).round(3)

### 6. Agent behaviour
Which tools each model used. Fine-tuning on plain RAG prompts can make a model worse at following the agent format, so check the format error rate here.

In [ ]:
from collections import Counter
usage = []
for path in sorted(glob.glob('runs/*_agent.jsonl')):
    rows = st.read_jsonl(path)
    c = Counter(step['tool'] for r in rows for step in r['trace'])
    usage.append({'run': os.path.basename(path)[:-6], **{t: round(n / len(rows), 2) for t, n in c.items()},
                  'format_error_rate': round(sum(r['format_errors'] > 0 for r in rows) / len(rows), 2)})
pd.DataFrame(usage).fillna(0)

### 7. Did the tools help?
The agent starts with the same chunks as standard RAG and chooses whether to call tools. So for each question we can compare the agent answer with the plain RAG answer from the same model.
- **used tools**: questions where the agent chose to read more. A positive `difference` means the extra reading gave a better summary.
- **no tools**: questions where it answered right away. The two answers should score about the same here.

In [ ]:
benefit = []
for path in sorted(glob.glob('runs/*_agent.jsonl')):
    rag_path = path.replace('_agent.jsonl', '_rag.jsonl')
    if os.path.exists(rag_path):
        benefit += [{'run': os.path.basename(path)[:-6], **row} for row in st.tool_benefit(path, rag_path)]
benefit = pd.DataFrame(benefit)
benefit.to_csv('results/tool_benefit.csv', index=False)
benefit.round(3)

Did it know when to use a tool? For specific questions we know if the starting chunks already held the gold evidence. A model with good judgment uses tools more often when the evidence was missed.

In [ ]:
decisions = pd.DataFrame([{'run': os.path.basename(p)[:-6], **row}
                          for p in sorted(glob.glob('runs/*_agent.jsonl')) for row in st.tool_decisions(p)])
decisions.to_csv('results/tool_decisions.csv', index=False)
decisions.round(3)

### 8. Error analysis
The final model's least faithful summaries, to see what kind of mistakes are left.

In [ ]:
FINAL = FINAL_RUN   # set in config.py
rows = st.read_jsonl(FINAL)
scores = faith.score([r['context'] for r in rows], [r['pred'] for r in rows])
for i in np.argsort(scores)[:3]:
    r = rows[i]
    print(f"Faithfulness {scores[i]:.2f} | {r['qtype']} | tools: {[s['tool'] for s in r['trace']]}")
    print('Q:', r['query']); print('Model:', r['pred']); print('Gold:', r['answer']); print('-' * 80)